# 01 — Data Preparation Walkthrough

This notebook turns the supplied `prepare_data.py` pipeline 

**Before running:** ensure these files exist:
- `data/raw/jigsaw/train.csv`
- `data/raw/sms/SMSSpamCollection`

Run cells from top to bottom. Outputs are written to `data/processed/`.

## 1. Import libraries

`Path` handles paths; pandas handles tables; NumPy provides missing-value support; `unicodedata` and `re` clean text; `json` writes a manifest; `train_test_split` creates data splits.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import unicodedata
import re
import json
from sklearn.model_selection import train_test_split

## 2. Find the project root

The original script uses `Path(__file__).resolve().parents[1]`, which works when the Python file is inside `src/`. Jupyter notebooks generally don't define `__file__`, so this cell searches the current directory and its parents for `data/raw/`.

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find project root containing data/raw/. "
        "Run this notebook from the project folder or one of its subfolders."
    )

ROOT = find_project_root(Path.cwd())
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw data folder:", RAW)
print("Processed output folder:", OUT)

Project root: /Users/kakulrajput/Documents/Group-03_Social-Community-app
Raw data folder: /Users/kakulrajput/Documents/Group-03_Social-Community-app/data/raw
Processed output folder: /Users/kakulrajput/Documents/Group-03_Social-Community-app/data/processed


## 3. Define labels and clean text

`clean_text`:
1. Returns an empty string for missing values.
2. Converts the value to a string and applies NFKC Unicode normalization.
3. Removes zero-width spaces (`U+200B`) and BOM characters (`U+FEFF`).
4. Collapses consecutive whitespace into one space and trims the ends.

Punctuation, letter case, and negation are preserved.

In [3]:
LABELS = [
    "toxic", "severe_toxic", "obscene",
    "threat", "insult", "identity_hate"
]

def clean_text(v):
    if pd.isna(v):
        return ""
    v = unicodedata.normalize("NFKC", str(v)).replace("\u200b", "").replace("\ufeff", "")
    return re.sub(r"\s+", " ", v).strip()

for example in ["  Hello   world  ", "ﬁne\u200b text", None]:
    print(repr(example), "->", repr(clean_text(example)))

'  Hello   world  ' -> 'Hello world'
'ﬁne\u200b text' -> 'fine text'
None -> ''


## 4. Load and clean Jigsaw toxicity data

Select the ID, comment text, and six toxicity labels. Clean the text, remove the original text column, drop blank/duplicate comments, and convert label values to `int8`. Spam-specific fields remain missing for Jigsaw rows.

In [4]:
jigsaw_path = RAW / "jigsaw" / "train.csv"
if not jigsaw_path.is_file():
    raise FileNotFoundError(f"Jigsaw file not found: {jigsaw_path}")

j = pd.read_csv(jigsaw_path)
j = j[["id", "comment_text"] + LABELS].copy()
j["text"] = j["comment_text"].map(clean_text)
j = j.drop(columns=["comment_text"])
j = j[j.text.ne("")].drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)

for c in LABELS:
    j[c] = pd.to_numeric(j[c], errors="coerce").fillna(0).astype("int8")

j["source"] = "jigsaw"
j["spam_label"] = np.nan
j["spam"] = np.nan
j = j[["id", "text", "source"] + LABELS + ["spam_label", "spam"]]

print("Cleaned Jigsaw rows:", len(j))
display(j.head())

Cleaned Jigsaw rows: 159374


,id,text,source,toxic,severe_toxic,obscene,threat,insult,identity_hate,spam_label,spam
0,0000997932d777bf,Explanation Why the edits made under my userna...,jigsaw,0,0,0,0,0,0,NaN,NaN
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,jigsaw,0,0,0,0,0,0,NaN,NaN
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",jigsaw,0,0,0,0,0,0,NaN,NaN
3,0001b41b1c6bb37e,""" More I can't make any real suggestions on im...",jigsaw,0,0,0,0,0,0,NaN,NaN
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",jigsaw,0,0,0,0,0,0,NaN,NaN


## 5. Load and clean SMS Spam Collection

Each valid line should contain a label and message separated by a tab. Only `ham` and `spam` are accepted. `split("\\t", 1)` splits only at the first tab, preserving any further tabs in the message.

In [5]:
sms_path = RAW / "sms" / "SMSSpamCollection"
if not sms_path.is_file():
    raise FileNotFoundError(f"SMS dataset file not found: {sms_path}")

rows = []
with open(sms_path, encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.rstrip("\r\n")
        if "\t" in line:
            label, text = line.split("\t", 1)
            label = label.strip().lower()
            if label in ("ham", "spam"):
                rows.append((text, label))

s = pd.DataFrame(rows, columns=["text", "spam_label"])
s["text"] = s.text.map(clean_text)
s = s[s.text.ne("")].drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)
s["id"] = [f"sms_{i:06d}" for i in range(len(s))]
s["source"] = "sms"
s["spam"] = (s.spam_label == "spam").astype("int8")

for c in LABELS:
    s[c] = np.nan

s = s[["id", "text", "source"] + LABELS + ["spam_label", "spam"]]
print("Cleaned SMS rows:", len(s))
display(s.head())
display(s["spam_label"].value_counts())

Cleaned SMS rows: 5160


,id,text,source,toxic,severe_toxic,obscene,threat,insult,identity_hate,spam_label,spam
0,sms_000000,"Go until jurong point, crazy.. Available only ...",sms,NaN,NaN,NaN,NaN,NaN,NaN,ham,0
1,sms_000001,Ok lar... Joking wif u oni...,sms,NaN,NaN,NaN,NaN,NaN,NaN,ham,0
2,sms_000002,Free entry in 2 a wkly comp to win FA Cup fina...,sms,NaN,NaN,NaN,NaN,NaN,NaN,spam,1
3,sms_000003,U dun say so early hor... U c already then say...,sms,NaN,NaN,NaN,NaN,NaN,NaN,ham,0
4,sms_000004,"Nah I don't think he goes to usf, he lives aro...",sms,NaN,NaN,NaN,NaN,NaN,NaN,ham,0


spam_label
ham     4518
spam     642
Name: count, dtype: int64

## 6. Combine datasets and save cleaned tables

The task-specific labels remain separate. Jigsaw rows have toxicity labels; SMS rows have spam labels. Missing labels are kept as null rather than interpreted as negative labels.

In [6]:
master = pd.concat([j, s], ignore_index=True)

j.to_csv(OUT / "jigsaw_cleaned.csv", index=False)
s.to_csv(OUT / "sms_cleaned.csv", index=False)
master.to_csv(OUT / "unified_master.csv", index=False)

print("Jigsaw rows:", len(j))
print("SMS rows:", len(s))
print("Unified master rows:", len(master))
display(master.head())

Jigsaw rows: 159374
SMS rows: 5160
Unified master rows: 164534


,id,text,source,toxic,severe_toxic,obscene,threat,insult,identity_hate,spam_label,spam
0,0000997932d777bf,Explanation Why the edits made under my userna...,jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,0001b41b1c6bb37e,""" More I can't make any real suggestions on im...",jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",jigsaw,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


## 7. Split Jigsaw data into train, validation, and test

The first split keeps 80% for training and 20% temporarily. The temporary 20% is split in half, giving approximately 80/10/10 overall. `random_state=42` makes the split reproducible. `stratify=j.toxic` preserves the toxic/non-toxic ratio as far as possible; it stratifies on the `toxic` label only.

In [7]:
jt, jtmp = train_test_split(
    j, test_size=0.2, random_state=42, stratify=j.toxic
)
jv, je = train_test_split(
    jtmp, test_size=0.5, random_state=42, stratify=jtmp.toxic
)

print("Jigsaw train:", len(jt))
print("Jigsaw validation:", len(jv))
print("Jigsaw test:", len(je))

Jigsaw train: 127499
Jigsaw validation: 15937
Jigsaw test: 15938


## 8. Split SMS data into train, validation, and test

The same 80/10/10 approach is used, stratifying by `spam_label` (`ham` or `spam`).

In [8]:
st, stmp = train_test_split(
    s, test_size=0.2, random_state=42, stratify=s.spam_label
)
sv, se = train_test_split(
    stmp, test_size=0.5, random_state=42, stratify=stmp.spam_label
)

print("SMS train:", len(st))
print("SMS validation:", len(sv))
print("SMS test:", len(se))

SMS train: 4128
SMS validation: 516
SMS test: 516


## 9. Save six split files

In [9]:
split_files = [
    (jt, "toxicity_train"),
    (jv, "toxicity_validation"),
    (je, "toxicity_test"),
    (st, "spam_train"),
    (sv, "spam_validation"),
    (se, "spam_test"),
]

for df, name in split_files:
    output_path = OUT / f"{name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {output_path.name}: {len(df)} rows")

Saved toxicity_train.csv: 127499 rows
Saved toxicity_validation.csv: 15937 rows
Saved toxicity_test.csv: 15938 rows
Saved spam_train.csv: 4128 rows
Saved spam_validation.csv: 516 rows
Saved spam_test.csv: 516 rows


## 10. Write the manifest

The manifest documents row counts and the normalization, label, and split policies used.

In [10]:
manifest = {
    "jigsaw_cleaned_rows": len(j),
    "sms_cleaned_rows": len(s),
    "unified_master_rows": len(master),
    "normalization": "NFKC; remove zero-width/BOM; collapse whitespace; trim; preserve punctuation, case, and negation",
    "label_policy": "Task labels remain separate; missing labels are null, not zero.",
    "split_policy": "Independent 80/10/10 split, random_state=42.",
}

manifest_path = OUT / "data_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

{
  "jigsaw_cleaned_rows": 159374,
  "sms_cleaned_rows": 5160,
  "unified_master_rows": 164534,
  "normalization": "NFKC; remove zero-width/BOM; collapse whitespace; trim; preserve punctuation, case, and negation",
  "label_policy": "Task labels remain separate; missing labels are null, not zero.",
  "split_policy": "Independent 80/10/10 split, random_state=42."
}


## 11. Verify generated files

If a source file is missing, check the dataset paths and folder names before rerunning.

In [11]:
expected_files = [
    "jigsaw_cleaned.csv",
    "sms_cleaned.csv",
    "unified_master.csv",
    "toxicity_train.csv",
    "toxicity_validation.csv",
    "toxicity_test.csv",
    "spam_train.csv",
    "spam_validation.csv",
    "spam_test.csv",
    "data_manifest.json",
]

for filename in expected_files:
    path = OUT / filename
    print(f"{'OK' if path.exists() else 'MISSING'}  {filename}")

OK  jigsaw_cleaned.csv
OK  sms_cleaned.csv
OK  unified_master.csv
OK  toxicity_train.csv
OK  toxicity_validation.csv
OK  toxicity_test.csv
OK  spam_train.csv
OK  spam_validation.csv
OK  spam_test.csv
OK  data_manifest.json
